# Esplorazione Rapida Lesioni: Riduzione Dimensionale e Clustering
Notebook interattivo per testare diverse metriche, configurazioni di embedding e algoritmi di clustering.

## 1. Setup e Caricamento Dati
Importiamo le librerie necessarie e carichiamo sia la matrice lesionale che i metadati clinici associati.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.analysis.distances import binary_pairwise_distance

%matplotlib inline

In [ ]:
# Caricamento Matrice
matrix_path = Path('../data/derived/lesion_matrix/21-07_s1.1/matrix.npy')
X = np.load(matrix_path)
print(f"Shape della matrice lesionale: {X.shape}")

In [ ]:
# Caricamento Metadata e Calcolo Volume
metadata_path = Path('../data/derived/lesion_matrix/21-07_s1.1/metadata.csv')
metadata = pd.read_csv(metadata_path)
metadata['volume_voxel'] = X.sum(axis=1)

# Arricchimento con variabili cliniche (Dataset orig, Lato, NIHSS)
clinical_meta_path = Path('../assets/metadata/UNIPD_WashU_participants_lesions.tsv')
clinical_meta = pd.read_csv(clinical_meta_path, sep='\t')
metadata = metadata.merge(clinical_meta, left_on='subject_id', right_on='participant_id', how='left')

print("Metadata caricati con successo!")
metadata[['subject_id', 'dataset_x', 'lesion_side', 'volume_voxel', 'NIHSS']].head()

## 2. Preparazione Matrice di Distanza
Calcoliamo le distanze pre-computate. **Nota:** per dati binari iper-sparsi come le lesioni (X voxel), metriche come `jaccard` o `dice` spesso collassano a 1.0 quando l'intersezione è vuota (es. lesioni in emisferi opposti).

In [ ]:
# Imposta la metrica desiderata: "euclidean", "jaccard", "dice"
metric = "jaccard" 

if metric in ["jaccard", "dice"]:
    print(f"Calcolo matrice di distanza ottimizzata per: {metric}")
    X_dist = binary_pairwise_distance(X, metric)
else:
    X_dist = X
    print(f"Nessun precalcolo richiesto, useremo la matrice originale con metrica {metric}")

In [ ]:
# Verifica Distribuzione Distanze
if metric in ['jaccard', 'dice']:
    plt.figure(figsize=(10, 5))
    dist_triu = X_dist[np.triu_indices_from(X_dist, k=1)]
    plt.hist(dist_triu, bins=50, color='purple', alpha=0.7)
    plt.title(f"Istogramma delle distanze Pairwise ({metric})")
    plt.xlabel("Distanza")
    plt.ylabel("Conteggio coppie (Soggetto-Soggetto)")
    plt.show()
else:
    print(f"L'istogramma è particolarmente utile per metriche non-euclidee. Metrica attuale: {metric}")

## 3. Riduzione Dimensionale: UMAP
Eseguiamo UMAP e analizziamo la separazione dello spazio latente in base ai metadati.

In [ ]:
# Parametri UMAP
n_neighbors = 15
n_components = 2
min_dist = 0.1
random_state = 42

reducer_umap = umap.UMAP(
    n_neighbors=n_neighbors,
    n_components=n_components,
    metric="precomputed" if metric in ["jaccard", "dice"] else metric,
    n_epochs=1000,
    learning_rate=1.0,
    init='spectral',
    min_dist=min_dist,
    spread=1.0,
    low_memory=False,
    set_op_mix_ratio=1.0,
    local_connectivity=1,
    repulsion_strength=1.0,
    negative_sample_rate=5,
    transform_queue_size=4.0,
    random_state=random_state
)

print("Fit UMAP in corso...")
embedding_umap = reducer_umap.fit_transform(X_dist)
print("Fatto.")

In [ ]:
# Plot UMAP
plt.figure(figsize=(7, 5))
plt.scatter(embedding_umap[:, 0], embedding_umap[:, 1], s=15, alpha=0.5, c='blue')
plt.title('UMAP - Spazio Latente Base')
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(x=embedding_umap[:, 0], y=embedding_umap[:, 1], hue=metadata['lesion_side'], palette='Set1', s=25, alpha=0.8)
plt.legend(title='Lato', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('UMAP - Raggruppamento per Emisfero')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
scatter = plt.scatter(x=embedding_umap[:, 0], y=embedding_umap[:, 1], c=metadata['volume_voxel'], cmap='magma', s=25, alpha=0.8)
plt.title('UMAP - Gradiente Volume Lesionale')
plt.colorbar(scatter, label='Voxel Lesionati')
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(x=embedding_umap[:, 0], y=embedding_umap[:, 1], hue=metadata['dataset_x'], palette='Set2', s=25, alpha=0.8)
plt.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('UMAP - Controllo Batch Effect (Dataset)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
scatter = plt.scatter(x=embedding_umap[:, 0], y=embedding_umap[:, 1], c=metadata['NIHSS'], cmap='viridis', s=25, alpha=0.8)
plt.title('UMAP - Severità NIHSS')
plt.colorbar(scatter, label='Score NIHSS')
plt.show()

## 4. Riduzione Dimensionale: t-SNE
Eseguiamo t-SNE e confrontiamolo con i risultati di UMAP.

In [ ]:
# Parametri t-SNE
perplexity = 30
random_state = 42

reducer_tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    metric="precomputed" if metric in ["jaccard", "dice"] else metric,
    init='pca' if metric not in ["jaccard", "dice"] else 'random',
    learning_rate='auto',
    random_state=random_state
)

print("Fit t-SNE in corso...")
embedding_tsne = reducer_tsne.fit_transform(X_dist)
print("Fatto.")

In [ ]:
# Plot t-SNE
plt.figure(figsize=(7, 5))
plt.scatter(embedding_tsne[:, 0], embedding_tsne[:, 1], s=15, alpha=0.5, c='green')
plt.title('t-SNE - Spazio Latente Base')
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(x=embedding_tsne[:, 0], y=embedding_tsne[:, 1], hue=metadata['lesion_side'], palette='Set1', s=25, alpha=0.8)
plt.legend(title='Lato', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('t-SNE - Raggruppamento per Emisfero')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
scatter = plt.scatter(x=embedding_tsne[:, 0], y=embedding_tsne[:, 1], c=metadata['volume_voxel'], cmap='magma', s=25, alpha=0.8)
plt.title('t-SNE - Gradiente Volume Lesionale')
plt.colorbar(scatter, label='Voxel Lesionati')
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(x=embedding_tsne[:, 0], y=embedding_tsne[:, 1], hue=metadata['dataset_x'], palette='Set2', s=25, alpha=0.8)
plt.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('t-SNE - Controllo Batch Effect (Dataset)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
scatter = plt.scatter(x=embedding_tsne[:, 0], y=embedding_tsne[:, 1], c=metadata['NIHSS'], cmap='viridis', s=25, alpha=0.8)
plt.title('t-SNE - Severità NIHSS')
plt.colorbar(scatter, label='Score NIHSS')
plt.show()

## 5. Clustering
Applichiamo algoritmi di partizionamento all'embedding selezionato (K-Means, Spectral, Agglomerative).

In [ ]:
# Selezione dell'embedding da usare per il clustering
embedding_da_usare = embedding_umap
# embedding_da_usare = embedding_tsne

print(f"L'embedding selezionato ha dimensioni: {embedding_da_usare.shape}")

### 5.1 K-Means

In [ ]:
k_kmeans = 4
kmeans = KMeans(n_clusters=k_kmeans, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(embedding_da_usare)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=embedding_da_usare[:, 0], y=embedding_da_usare[:, 1], hue=labels_kmeans, palette='tab10', legend='full', s=25, alpha=0.8)
plt.title(f'K-Means (k={k_kmeans})')
plt.show()

### 5.2 Spectral Clustering

In [ ]:
k_spectral = 4
spectral = SpectralClustering(n_clusters=k_spectral, affinity='nearest_neighbors', random_state=42)
labels_spectral = spectral.fit_predict(embedding_da_usare)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=embedding_da_usare[:, 0], y=embedding_da_usare[:, 1], hue=labels_spectral, palette='tab10', legend='full', s=25, alpha=0.8)
plt.title(f'Spectral Clustering (k={k_spectral})')
plt.show()

### 5.3 Agglomerative Clustering

In [ ]:
k_agglomerative = 4
agglomerative = AgglomerativeClustering(n_clusters=k_agglomerative)
labels_agglomerative = agglomerative.fit_predict(embedding_da_usare)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=embedding_da_usare[:, 0], y=embedding_da_usare[:, 1], hue=labels_agglomerative, palette='tab10', legend='full', s=25, alpha=0.8)
plt.title(f'Agglomerative Clustering (k={k_agglomerative})')
plt.show()